<a href="https://colab.research.google.com/github/ishitathakwani7265/ML/blob/main/practical3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA

In [ ]:
df = pd.read_csv("/content/01 covid_toy.csv")

print(df.dtypes)

age            int64
gender        object
fever        float64
cough         object
city          object
has_covid     object
dtype: object


In [ ]:
print(df.select_dtypes(include=np.number).columns)

print(df.select_dtypes(include="object").columns)

display(df.head())

Index(['age', 'fever'], dtype='object')
Index(['gender', 'cough', 'city', 'has_covid'], dtype='object')


,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [ ]:
df = df.replace("?", np.nan)

df["has_covid"] = df["has_covid"].astype("object")

print(df.isnull().sum())

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64


In [ ]:
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["fever"] = pd.to_numeric(df["fever"], errors="coerce")

In [ ]:
numeric_features = df.select_dtypes(include=np.number)

imputer = SimpleImputer(strategy="mean")

numeric_features = pd.DataFrame(
    imputer.fit_transform(numeric_features),
    columns=numeric_features.columns,
    index=numeric_features.index
)

display(numeric_features.head())

,age,fever
0,60.0,103.0
1,27.0,100.0
2,42.0,101.0
3,31.0,98.0
4,65.0,101.0


In [ ]:
scaler = MinMaxScaler().set_output(transform="pandas")

scaled_data = scaler.fit_transform(numeric_features)

display(scaled_data.head())

,age,fever
0,0.696203,0.833333
1,0.278481,0.333333
2,0.468354,0.500000
3,0.329114,0.000000
4,0.759494,0.500000


In [ ]:
categorical_features = df.select_dtypes(include=["object"])

for column in categorical_features.columns:
    if column != "has_covid":
        categorical_features[column] = categorical_features[column].fillna(
            categorical_features[column].mode().iloc[0]
            if not categorical_features[column].mode().empty
            else "Unknown"
        )

In [ ]:
categorical_input = categorical_features.drop(
    columns=["has_covid"]
)

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
).set_output(transform="pandas")

encoded_data = encoder.fit_transform(categorical_input)

display(encoded_data.head())

,gender_Female,gender_Male,cough_Mild,cough_Strong,city_Bangalore,city_Delhi,city_Kolkata,city_Mumbai
0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
1,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
2,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
3,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


In [ ]:
processed_features = pd.concat(
    [scaled_data, encoded_data],
    axis=1
)

display(processed_features.head())

,age,fever,gender_Female,gender_Male,cough_Mild,cough_Strong,city_Bangalore,city_Delhi,city_Kolkata,city_Mumbai
0,0.696203,0.833333,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
1,0.278481,0.333333,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
2,0.468354,0.500000,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
3,0.329114,0.000000,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,0.759494,0.500000,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


In [ ]:
target = (
    df["has_covid"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({"no": 0, "yes": 1})
)

valid_rows = target.notna()

processed_features = processed_features.loc[valid_rows]
target = target.loc[valid_rows].astype(int)

In [ ]:
selector = SelectKBest(
    score_func=f_classif,
    k=3
).set_output(transform="pandas")

selected_features = selector.fit_transform(
    processed_features,
    target
)

display(selected_features.head())

,fever,cough_Mild,cough_Strong
0,0.833333,1.0,0.0
1,0.333333,1.0,0.0
2,0.500000,1.0,0.0
3,0.000000,1.0,0.0
4,0.500000,1.0,0.0


In [ ]:
feature_scores = pd.DataFrame({
    "Feature": processed_features.columns,
    "Score": selector.scores_
}).sort_values(
    by="Score",
    ascending=False
)

display(feature_scores)

,Feature,Score
1,fever,3.176506
5,cough_Strong,0.746812
4,cough_Mild,0.746812
3,gender_Male,0.394843
2,gender_Female,0.394843
9,city_Mumbai,0.188916
7,city_Delhi,0.187261
0,age,0.069993
8,city_Kolkata,0.065552
6,city_Bangalore,0.047161


In [ ]:
standard_scaler = StandardScaler().set_output(
    transform="pandas"
)

standardized_features = standard_scaler.fit_transform(
    selected_features
)

display(standardized_features.head())

,fever,cough_Mild,cough_Strong
0,1.111906,0.782881,-0.782881
1,-0.435592,0.782881,-0.782881
2,0.080241,0.782881,-0.782881
3,-1.467257,0.782881,-0.782881
4,0.080241,0.782881,-0.782881


In [ ]:
pca = PCA(n_components=2)

pca_result = pca.fit_transform(standardized_features)

pca_data = pd.DataFrame(
    pca_result,
    columns=["PC1", "PC2"],
    index=standardized_features.index
)

display(pca_data.head())

,PC1,PC2
0,1.292370,0.889899
1,1.008449,-0.631330
2,1.103089,-0.124254
3,0.819168,-1.645483
4,1.103089,-0.124254


In [ ]:
print(pca.explained_variance_ratio_)

print(pca.explained_variance_ratio_.sum())

[0.6786972 0.3213028]
1.0
